### Tool intergration

In [21]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv('OPENAI_API_KEY')
os.environ["GEMINI_API_KEY"]=os.getenv('GEMINI_API_KEY')
os.environ["GROQ_API_KEY"]=os.getenv('GROQ_API')

In [22]:
from langchain.chat_models import init_chat_model
model = init_chat_model(model="groq:qwen/qwen3-32b")
response=model.invoke("why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots and their ability to mimic speech. First off, parrots are part of the Psittaciformes order, and they\'re known for their vocal mimicry skills. But why do they do that? Well, I remember that in the wild, parrots use a lot of vocalizations to communicate with each other, like calling to mates or warning about predators. So maybe talking in captivity is an extension of that natural behavior.\n\nI think one reason is social bonding. Parrots are social animals, so in the wild, they might imitate sounds to stay in contact with their flock. When they\'re kept as pets, they might mimic their owners\' voices to bond with them. That makes sense because they see the owner as part of their social group. So talking could be a way to interact and feel connected.\n\nAnother angle is survival. In the wild, parrots learn to mimic sounds from their environment—other birds, animal

In [23]:
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get weater at a location"""
    return f"It's sunny in {location}"

model_with_tool= model.bind_tools([get_weather])



In [24]:
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002740361B9D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000274036A4410>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [25]:
model.invoke("weater in lahore")

AIMessage(content='<think>\nOkay, the user is asking for the weather in Lahore. Since I don\'t have real-time data, I need to explain that first. Let me check the current standard response for such queries.\n\nI should mention that I can\'t provide live weather info and suggest checking a weather website or app. Maybe give examples like Weather.com or AccuWeather. Also, offer to describe the typical weather in Lahore based on the current time of year.\n\nWait, what\'s the typical weather like in Lahore now? Let me think. Lahore is in Pakistan, has a subtropical climate. If it\'s currently summer, it\'s hot and dry. If it\'s winter, maybe cooler but still warm. But I should be cautious not to assume the exact month. Maybe just say that Lahore usually has hot summers and mild winters, and suggest checking for the current conditions.\n\nAlso, maybe mention that they can use a search engine for the most up-to-date info. Keep the response friendly and helpful. Let me put that all together c

In [26]:
response=model_with_tool.invoke("what's the weather in lohore")

In [27]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Lahore, Pakistan'},
  'id': '0s4ens4zw',
  'type': 'tool_call'}]

### Tool Execution loop

In [ ]:
messages =[{
    "role":"user", 
    "content":"what's the weater of Lahore?"
}]


ai_message= model_with_tool.invoke(messages)

for tool_call in ai_message.tool_calls:
    tool_result= get_weather.invoke(tool_call)
    messages.append(tool_result)

model_with_tool


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002740361B9D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000274036A4410>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get weater at a location', 'paramet

In [29]:
final_response = model_with_tool.invoke(messages)
final_response.text    

'The weather in Lahore is sunny.'

In [30]:
messages

[{'role': 'user', 'content': "what's the weater of Lahore?"},
 ToolMessage(content="It's sunny in Lahore", name='get_weather', tool_call_id='0g7yjth82')]